In [1]:
from dotenv import load_dotenv
load_dotenv("./../.env")

True

In [2]:
# configurations
DATA_DIR = "data"
CHROMA_DIR = "./chroma_financial_db"
COLLECTION_NAME = "financial_docs"
EMBEDDING_MODEL = 'nomic-embed-text'
BASE_URL = 'http://localhost:11434'
NUM_CTX = 8192 # 向量化维度

In [3]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=BASE_URL, num_ctx=NUM_CTX)

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)

In [4]:
processed_hashes = set()

Ingest page data into database

In [6]:
from utils import page_ingest
from pathlib import Path

files = ["data/amazon/amazon 10-k 2023.pdf", "data/apple/apple 8-k q4 2023.pdf", "data/google/google 10-k 2023.pdf"]

for path in files:
    pdf_path = Path(path)
    page_ingest.ingest_docs_in_vectordb(pdf_path=pdf_path, vector_store=vector_store, processed_hashes=processed_hashes)
    print(f'finished ingest file {path} into database')

Processing: amazon 10-k 2023.pdf


[INFO] 2026-03-10 22:36:42,846 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-10 22:36:42,852 [RapidOCR] download_file.py:60: File exists and is valid: D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-10 22:36:42,853 [RapidOCR] main.py:53: Using D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-10 22:36:42,934 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-10 22:36:42,936 [RapidOCR] download_file.py:60: File exists and is valid: D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-10 22:36:42,936 [RapidOCR] main.py:53: Using D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-10 22:36:42,977 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-10 22:36:42,983 [RapidOCR] download_file.py:60

finished ingest file data/amazon/amazon 10-k 2023.pdf into database
Processing: apple 8-k q4 2023.pdf


[INFO] 2026-03-10 22:37:37,360 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-10 22:37:37,364 [RapidOCR] download_file.py:60: File exists and is valid: D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-10 22:37:37,364 [RapidOCR] main.py:53: Using D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-10 22:37:37,440 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-10 22:37:37,442 [RapidOCR] download_file.py:60: File exists and is valid: D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-10 22:37:37,442 [RapidOCR] main.py:53: Using D:\Anaconda\envs\langchainv1.0\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-10 22:37:37,483 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-10 22:37:37,491 [RapidOCR] download_file.py:60

finished ingest file data/apple/apple 8-k q4 2023.pdf into database
Processing: google 10-k 2023.pdf
finished ingest file data/google/google 10-k 2023.pdf into database


In [7]:
vector_store._collection.count()

213

In [11]:
existing_docs = vector_store.get(where={"file_hash": {"$ne": ""}}, include=['metadatas'])
processed_hashes = [m.get('file_hash') for m in existing_docs['metadatas'] if m.get('file_hash')]
processed_hashes = set(processed_hashes)
processed_hashes

{'1dceed8775b7b56432618aff917e08f9fe9573921f397a79b14bad5ab31e469a',
 '6e5549c7b20b0fbc5f482397070a1e85cbf8643c801ff570903f52366b11154f',
 'bfb57cd34d8c3d9f650b54f4914d712f7d88d57aa7b374886e23101da3228356'}

In [13]:
page_ingest.ingest_docs_in_vectordb(pdf_path=Path("data/apple/apple 8-k q4 2023.pdf"),vector_store=vector_store, processed_hashes=processed_hashes)

Processing: apple 8-k q4 2023.pdf
[SKIP] already processed: data\apple\apple 8-k q4 2023.pdf


In [14]:
# 关键词搜索
vector_store.get(where={"company_name": "amazon"}, limit=3)

{'ids': ['051f2c18-3bd4-444e-bb2f-e2b8b690a832',
  '58e91bd6-26ad-43ba-af6f-f44c8b204318',
  '681db586-a33f-4634-a1db-8b8e323c9977'],
 'embeddings': None,
 'documents': ["## UNITED STATES\n\n## SECURITIES AND EXCHANGE COMMISSION\n\nWashington, D.C. 20549\n\n\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\n\nFORM 10-K\n\n\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\n\n(Mark One)\n\n- [x] ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\n\nFor the fiscal year ended December 31, 2023\n\nor\n\n- [ ] ☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\n\nFor the transition period from            to             .\n\nCommission File No. 000-22513\n\n\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\\_\n\n## AMAZON.COM, INC.\n\n(Exact name 

In [15]:
# 相似度搜索
results = vector_store.search("What is Apple's revenue for Q1 2024", search_type="similarity")
results

[Document(id='8c5d39cf-f7ad-4e08-bf09-caa0ec8d4b46', metadata={'file_hash': 'bfb57cd34d8c3d9f650b54f4914d712f7d88d57aa7b374886e23101da3228356', 'company_name': 'apple', 'fiscal_quarter': 'q4', 'page': 5, 'source_file': 'apple 8-k q4 2023.pdf', 'doc_type': '8-k', 'fiscal_year': 2023}, page_content="\n\n## Apple reports first quarter results\n\n## Services revenue reaches new all-time record\n\n## EPS up 16 percent to new all-time high\n\nCUPERTINO, CALIFORNIA - Apple  today announced financial results for its fiscal 2024 first quarter ended December 30, 2023. The Company posted quarterly revenue of $119.6 billion, up 2 percent year over year, and quarterly earnings per diluted share of $2.18, up 16 percent year over year. ®\n\n'Today Apple is reporting revenue growth for the December quarter fueled by iPhone sales, and an all-time revenue record in Services,' said Tim Cook, Apple's CEO. 'We are pleased to announce that our installed base of active devices has now surpassed 2.2 billion, 